<a href="https://colab.research.google.com/github/SNayak-166/Evaluation_of_RecommenderSystem/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell-1: Loading data from Kaggle. ratings.csv data is of >600mb. GitHub do not allow >25mb files so directly downloading from Kaggle.

Requirement to download: Kaggle username and API key

In [1]:
import os
import getpass
import shutil
import pandas as pd

destination_path = "ratings.csv"

# Check if ratings.csv is already available in the local Colab workspace
if os.path.exists(destination_path):
    print("--- ratings.csv already exists locally in workspace. Skipping download. ---")
    ratings_df = pd.read_csv(destination_path)
    print(f"File location: {os.path.abspath(destination_path)}")
else:
    print("--- ratings.csv not found locally. Downloading from Kaggle... ---")

    # 1. Securely ask the evaluator for their Kaggle credentials
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("--- Kaggle Credentials Required ---")
        os.environ["KAGGLE_USERNAME"] = input("Enter your Kaggle Username: ")
        os.environ["KAGGLE_KEY"] = getpass.getpass("Enter your Kaggle API Key (input will be hidden): ")

    # 2. Install and import kagglehub
    !pip install kagglehub --quiet
    import kagglehub

    # 3. Download the dataset
    path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")
    print("Dataset downloaded to:", path)

    # 4. Copy ratings.csv to the local Colab workspace for easy access
    source_path = os.path.join(path, "ratings.csv")
    shutil.copy(source_path, destination_path)

    # 5. Load ratings_df from the workspace file
    ratings_df = pd.read_csv(destination_path)
    print("ratings.csv successfully downloaded, saved locally, and loaded!")
    print(f"File location: {os.path.abspath(destination_path)}")

--- ratings.csv already exists locally in workspace. Skipping download. ---
File location: /content/ratings.csv


Cell-2: Storing the processed_movies in the disk.

In [2]:
import pandas as pd
import os

# Replace with your actual GitHub raw file URL
github_url = "https://raw.githubusercontent.com/SNayak-166/Evaluation_of_RecommenderSystem/refs/heads/genreencoding/processed_movies.csv"

# Load the processed movies dataframe directly from the URL
processed_movies_df = pd.read_csv(github_url)

# Save a local copy of processed_movies.csv in the Google Colab VM workspace
local_path = "processed_movies.csv"
processed_movies_df.to_csv(local_path, index=False)

print(f"processed_movies.csv successfully saved to Colab VM!")
print(f"File location: {os.path.abspath(local_path)}")

processed_movies.csv successfully saved to Colab VM!
File location: /content/processed_movies.csv


Cell-3: This cell splits the consolidated interaction dataset (`ratings_df`) into a synchronized 67% training set and a 33% testing set using a fixed random seed. Both splits are stored directly on the local VM disk to conserve system RAM, enabling independent downstream models to draw from identical training profiles.

In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os
import gc

train_csv = 'train_ratings.csv'
test_csv = 'test_ratings.csv'

if os.path.exists(train_csv) and os.path.exists(test_csv):
    print(f"--- Both '{train_csv}' and '{test_csv}' already exist on disk. Skipping split. ---")
else:
    print("--- Splitting ratings_df into synchronized train/test (67/33) sets... ---")
    train_df, test_df = train_test_split(ratings_df, test_size=0.33, random_state=42)

    print("Saving splits to disk as CSV files...")
    # Note: We split ratings only to save disk space and prevent redundancy.
    # The genre features from processed_movies_df are merged on-the-fly during model training (Step 2).
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)

    print("Splits successfully written to disk!")
    print(f"-> Train Set size: {len(train_df):,} samples (saved to 'train_ratings.csv')")
    print(f"-> Test Set size: {len(test_df):,} samples (saved to 'test_ratings.csv')")
    print("Note: Genre features will be joined dynamically on 'movieId' during training to conserve RAM and storage.")

    del train_df, test_df
    gc.collect()
    print("RAM cleared of split variables!")

--- Both 'train_ratings.csv' and 'test_ratings.csv' already exist on disk. Skipping split. ---


### Model Initialization Option: Train or Skip and Download
Run this cell to decide whether to train all models from scratch or skip the training phase entirely by downloading pre-trained weights from GitHub.

In [4]:
# @title Select Mode { run: "auto" }
EXECUTION_MODE = "Train From Scratch" # @param ["Train From Scratch", "Download Pre-trained Models (Skip Training)"]

import os
import subprocess

if EXECUTION_MODE == "Download Pre-trained Models (Skip Training)":
    print("--- Mode: SKIP TRAINING (Downloading Pre-trained Models from Google Drive) ---")

    # Google Drive public folder containing the large pre-trained model files
    gdrive_folder_url = "https://drive.google.com/drive/folders/1p0uzqSqgQDHk2t8SVYKtG1m8SjEyH3cb?usp=drive_link"

    # Install/upgrade gdown to ensure folder downloads work reliably
    try:
        import gdown
    except ImportError:
        print("Installing gdown...")
        subprocess.run(["pip", "install", "--upgrade", "gdown", "--quiet"])
        import gdown

    # Download the entire folder contents to the workspace root
    print("Downloading artifacts from Google Drive folder...")
    gdown.download_folder(url=gdrive_folder_url, remaining_ok=True, quiet=False)

    print("\nAll pre-trained model artifacts are ready for Evaluation! You can skip Cell-4, Cell-5, and Cell-6.")
else:
    print("--- Mode: TRAIN FROM SCRATCH ---")
    print("Please proceed to execute Cell-4, Cell-5, and Cell-6 sequentially to generate new models.")

--- Mode: TRAIN FROM SCRATCH ---
Please proceed to execute Cell-4, Cell-5, and Cell-6 sequentially to generate new models.


Cell-4: This cell incrementally trains both **Naive Bayes** and **SGD Logistic Regression** models. It reads the shared `train_ratings.csv` in memory-efficient chunks, maps movie genres, fits the models, and persists them directly to disk as `.pkl` files.

In [5]:
import gc
import joblib
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
import pandas as pd
import numpy as np

# Check if we should skip training
if 'EXECUTION_MODE' in globals() and EXECUTION_MODE == "Download Pre-trained Models (Skip Training)":
    print("--- Mode is set to Skip Training. Skipping Content-Based training cell! ---")
else:
    # Define the static movie genre feature columns directly inside the training cell
    genre_columns = [
        '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]

    # Optimized hyperparameters based on playground sweep: alpha=0.0001, penalty='elasticnet'
    lr_sgd = SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=0.0001, l1_ratio=0.15, random_state=42)
    nb_model = MultinomialNB()

    chunk_size = 150000
    print("Starting multi-batch training on Content-Based models using genre features (Optimized)...")

    for chunk_idx, chunk in enumerate(pd.read_csv('train_ratings.csv', chunksize=chunk_size)):
        # Join interactions chunk with the local processed_movies metadata dynamically on movieId
        chunk_merged = pd.merge(chunk, processed_movies_df, on='movieId')
        chunk_merged['liked'] = (chunk_merged['rating'] >= 4.0).astype(int)

        X_chunk = chunk_merged[genre_columns]
        y_chunk = chunk_merged['liked']

        if chunk_idx == 0:
            nb_model.partial_fit(X_chunk, y_chunk, classes=np.array([0, 1]))
        else:
            nb_model.partial_fit(X_chunk, y_chunk)

        lr_sgd.partial_fit(X_chunk, y_chunk, classes=np.array([0, 1]))

        del chunk, chunk_merged, X_chunk, y_chunk
        gc.collect()

    print("Saving Content-Based models to disk...")
    joblib.dump(nb_model, 'nb_model.pkl')
    joblib.dump(lr_sgd, 'lr_sgd.pkl')

    del nb_model, lr_sgd
    gc.collect()
    print("Content-Based models saved and RAM cleared!")

Starting multi-batch training on Content-Based models using genre features (Optimized)...
Saving Content-Based models to disk...
Content-Based models saved and RAM cleared!


Cell-5: This cell reads our standardized training CSV data file into the `surprise` dataset framework and trains an SVD (Singular Value Decomposition) Collaborative Filtering model. The model is saved to disk immediately to free up active memory.

In [6]:
import sys
import subprocess
import os
import gc
import joblib
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

# Check if we should skip training
if 'EXECUTION_MODE' in globals() and EXECUTION_MODE == "Download Pre-trained Models (Skip Training)":
    print("--- Mode is set to Skip Training. Skipping ALS Collaborative Filtering training cell! ---")
else:
    try:
        import implicit
    except ImportError:
        print("--- 'implicit' library not found. Installing dynamically... ---")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "implicit"])
        import implicit

    train_csv_path = 'train_ratings.csv'

    print("Loading full training ratings into pandas (optimizing datatypes for memory)...")
    # Load the full dataset (16.7+ million rows) but only use needed columns with memory-efficient dtypes
    train_df = pd.read_csv(
        train_csv_path,
        usecols=['userId', 'movieId', 'rating'],
        dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'}
    )

    # Convert userId and movieId to categoricals so we can build a compact sparse matrix
    print("Creating category mappings...")
    train_df['user_cat'] = train_df['userId'].astype('category')
    train_df['movie_cat'] = train_df['movieId'].astype('category')

    # Keep references to categoricals so we can reconstruct mapping during evaluation
    user_categories = train_df['user_cat'].cat.categories
    movie_categories = train_df['movie_cat'].cat.categories

    # Construct the Sparse CSR Matrix
    print("Building the sparse user-item interaction matrix...")
    user_items = csr_matrix(
        (train_df['rating'], (train_df['user_cat'].cat.codes, train_df['movie_cat'].cat.codes)),
        shape=(len(user_categories), len(movie_categories))
    )

    # Clear raw dataframe from RAM immediately
    del train_df
    gc.collect()

    print("Training Alternating Least Squares (ALS) model with Optimized Factors...")
    # Increased factors to 64 for finer latent representation capture
    als_model = implicit.als.AlternatingLeastSquares(
        factors=64,
        regularization=0.05,
        iterations=15,
        random_state=42
    )
    als_model.fit(user_items)

    print("Saving trained ALS model and category maps to disk...")
    joblib.dump({
        'model': als_model,
        'user_categories': user_categories,
        'movie_categories': movie_categories
    }, 'als_model.pkl')

    # Cleanup to prevent evaluation step memory issues
    del user_items, als_model, user_categories, movie_categories
    gc.collect()
    print("ALS model successfully trained on the FULL dataset and saved to disk!")

Loading full training ratings into pandas (optimizing datatypes for memory)...
Creating category mappings...
Building the sparse user-item interaction matrix...
Training Alternating Least Squares (ALS) model with Optimized Factors...


/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Saving trained ALS model and category maps to disk...
ALS model successfully trained on the FULL dataset and saved to disk!


Cell-6: This cell trains a Sparse Linear Methods (SLIM) model over the standardized training set. To protect system RAM limits, it solves regression models for a subset of the target items, writes the learned similarity weight matrix and ID dictionaries to CSVs, and frees all memory structures.

In [8]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import ElasticNet
from joblib import Parallel, delayed
import os
import gc

# Check if we should skip training
if 'EXECUTION_MODE' in globals() and EXECUTION_MODE == "Download Pre-trained Models (Skip Training)":
    print("--- Mode is set to Skip Training. Skipping SLIM training cell! ---")
else:
    print("Loading train_ratings.csv for SLIM...")
    train_ratings = pd.read_csv('train_ratings.csv')

    n_users = train_ratings['userId'].nunique()
    n_movies = train_ratings['movieId'].nunique()

    users = train_ratings['userId'].astype('category')
    movies = train_ratings['movieId'].astype('category')

    user_item_matrix = csr_matrix(
        (train_ratings['rating'], (users.cat.codes, movies.cat.codes)),
        shape=(n_users, n_movies)
    ).tocsc()

    num_items = user_item_matrix.shape[1]
    max_items_to_train = min(num_items, 500)

    # Define the training function for a single item to run in parallel
    def train_single_item(j, user_item_matrix):
        y_col = user_item_matrix[:, j].toarray().ravel()
        # Exclude the target item j from its own features
        left_cols = user_item_matrix[:, :j]
        right_cols = user_item_matrix[:, j+1:]
        X_cols = hstack([left_cols, right_cols]).tocsr()

        # Train ElasticNet with your exact optimized configurations
        en = ElasticNet(alpha=0.01, l1_ratio=0.15, positive=True, fit_intercept=False, max_iter=300)
        en.fit(X_cols, y_col)
        return j, en.coef_

    print(f"Training SLIM model in PARALLEL (using all CPU cores) with Optimized Regularization (alpha=0.01) on {max_items_to_train} target items...")

    # Run parallel jobs across available CPU threads
    results = Parallel(n_jobs=-1, batch_size=10)(
        delayed(train_single_item)(j, user_item_matrix) for j in range(max_items_to_train)
    )

    # Assemble the weight matrix W
    W = np.zeros((max_items_to_train, num_items))
    for j, coefs in results:
        W[j, :j] = coefs[:j]
        W[j, j+1:] = coefs[j:]

    print("Saving SLIM weights and mappings to disk...")
    pd.DataFrame(W).to_csv("slim_weights.csv", index=False, header=False)

    user_map = {uid: code for code, uid in enumerate(users.cat.categories)}
    movie_map = {mid: code for code, mid in enumerate(movies.cat.categories)}
    pd.DataFrame(list(user_map.items()), columns=['userId', 'code']).to_csv("slim_user_map.csv", index=False)
    pd.DataFrame(list(movie_map.items()), columns=['movieId', 'code']).to_csv("slim_movie_map.csv", index=False)

    del train_ratings, user_item_matrix, W, user_map, movie_map, users, movies, results
    gc.collect()
    print("Parallel SLIM training finalized and RAM cleared!")

Loading train_ratings.csv for SLIM...
Training SLIM model in PARALLEL (using all CPU cores) with Optimized Regularization (alpha=0.01) on 500 target items...
Saving SLIM weights and mappings to disk...
Parallel SLIM training finalized and RAM cleared!


Cell-7: unified evaluation code that evaluates all four models on the exact same footing—outputting RMSE, MAE, Accuracy, Precision, Recall, F1-Score, and ROC-AUC for everyone

In [23]:
from sklearn.metrics import classification_report, roc_auc_score, mean_squared_error, mean_absolute_error, confusion_matrix
import numpy as np
import pandas as pd
import joblib
import gc
import os
from scipy.sparse import csr_matrix

print("=== STARTING MEMORY-OPTIMIZED SERIAL EVALUATION PHASE ===")

evaluation_summary = {}

# Helper function to calculate False Positive Rate
def calculate_fpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return fpr

# -------------------------------------------------------------
# 1. EVALUATE CONTENT-BASED MODELS (Chunk-by-Chunk to prevent OOM)
# -----------------------------------------------------
print("\n--- [Step 1/3] Evaluating Content-Based Models in chunks ---")
if os.path.exists('nb_model.pkl') and os.path.exists('lr_sgd.pkl') and os.path.exists('processed_movies.csv'):
    print("Loading Naive Bayes and Logistic Regression models...")
    nb_model = joblib.load('nb_model.pkl')
    lr_sgd = joblib.load('lr_sgd.pkl')
    processed_movies_df = pd.read_csv('processed_movies.csv')

    genre_columns = [
        '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]

    # Initialize tracking variables
    nb_y_true, nb_y_pred, nb_y_probs = [], [], []
    lr_y_true, lr_y_pred, lr_y_probs = [], [], []
    y_true_ratings = []

    chunk_size = 500000
    print("Reading test_ratings.csv in chunks...")
    for chunk in pd.read_csv('test_ratings.csv', chunksize=chunk_size):
        chunk_merged = pd.merge(chunk, processed_movies_df, on='movieId')
        chunk_merged['liked'] = (chunk_merged['rating'] >= 4.0).astype(int)

        X_chunk = chunk_merged[genre_columns]
        y_chunk_bin = chunk_merged['liked'].values
        y_chunk_raw = chunk_merged['rating'].values

        # Collect actuals
        nb_y_true.extend(y_chunk_bin)
        y_true_ratings.extend(y_chunk_raw)

        # Naive Bayes Predictions
        nb_y_pred.extend(nb_model.predict(X_chunk))
        try:
            nb_y_probs.extend(nb_model.predict_proba(X_chunk)[:, 1])
        except:
            pass

        # Logistic Regression Predictions
        lr_y_true.extend(y_chunk_bin)
        lr_y_pred.extend(lr_sgd.predict(X_chunk))
        try:
            lr_y_probs.extend(lr_sgd.predict_proba(X_chunk)[:, 1])
        except:
            pass

        del chunk, chunk_merged, X_chunk
        gc.collect()

    # Evaluate Naive Bayes
    print("Compiling Naive Bayes Metrics...")
    nb_report = classification_report(nb_y_true, nb_y_pred, output_dict=True, zero_division=0)
    nb_auc = roc_auc_score(nb_y_true, nb_y_probs) if nb_y_probs else "N/A"
    nb_mapped = 0.5 + (np.array(nb_y_probs) * 4.5) if nb_y_probs else None
    nb_rmse = np.sqrt(mean_squared_error(y_true_ratings, nb_mapped)) if nb_mapped is not None else "N/A"
    nb_mae = mean_absolute_error(y_true_ratings, nb_mapped) if nb_mapped is not None else "N/A"
    nb_fpr = calculate_fpr(nb_y_true, nb_y_pred)

    evaluation_summary["Naive Bayes"] = {
        "Accuracy": round(nb_report['accuracy'], 4),
        "Precision (Liked)": round(nb_report['1']['precision'], 4),
        "Recall (Liked)": round(nb_report['1']['recall'], 4),
        "F1-Score (Liked)": round(nb_report['1']['f1-score'], 4),
        "ROC-AUC": round(nb_auc, 4) if isinstance(nb_auc, float) else nb_auc,
        "FPR": round(nb_fpr, 4),
        "RMSE": round(nb_rmse, 4) if isinstance(nb_rmse, float) else nb_rmse,
        "MAE": round(nb_mae, 4) if isinstance(nb_mae, float) else nb_mae
    }

    # Evaluate Logistic Regression
    print("Compiling Logistic Regression Metrics...")
    lr_report = classification_report(lr_y_true, lr_y_pred, output_dict=True, zero_division=0)
    lr_auc = roc_auc_score(lr_y_true, lr_y_probs) if lr_y_probs else "N/A"
    lr_mapped = 0.5 + (np.array(lr_y_probs) * 4.5) if lr_y_probs else None
    lr_rmse = np.sqrt(mean_squared_error(y_true_ratings, lr_mapped)) if lr_mapped is not None else "N/A"
    lr_mae = mean_absolute_error(y_true_ratings, lr_mapped) if lr_mapped is not None else "N/A"
    lr_fpr = calculate_fpr(lr_y_true, lr_y_pred)

    evaluation_summary["Logistic Regression"] = {
        "Accuracy": round(lr_report['accuracy'], 4),
        "Precision (Liked)": round(lr_report['1']['precision'], 4),
        "Recall (Liked)": round(lr_report['1']['recall'], 4),
        "F1-Score (Liked)": round(lr_report['1']['f1-score'], 4),
        "ROC-AUC": round(lr_auc, 4) if isinstance(lr_auc, float) else lr_auc,
        "FPR": round(lr_fpr, 4),
        "RMSE": round(lr_rmse, 4) if isinstance(lr_rmse, float) else lr_rmse,
        "MAE": round(lr_mae, 4) if isinstance(lr_mae, float) else lr_mae
    }

    # Severe purge
    del nb_model, lr_sgd, processed_movies_df
    del nb_y_true, nb_y_pred, nb_y_probs, lr_y_true, lr_y_pred, lr_y_probs, y_true_ratings
    gc.collect()
else:
    print("Content-Based files or model pickles missing.")

# -------------------------------------------------------------
# 2. EVALUATE ALS COLLABORATIVE FILTERING
# -------------------------------------------------------------
print("\n--- [Step 2/3] Evaluating ALS Model ---")
if os.path.exists('als_model.pkl') and os.path.exists('test_ratings.csv'):
    print("Loading ALS Model files...")
    als_data = joblib.load('als_model.pkl')
    als_model = als_data['model']
    user_categories = als_data['user_categories']
    movie_categories = als_data['movie_categories']

    user_to_code = {uid: idx for idx, uid in enumerate(user_categories)}
    movie_to_code = {mid: idx for idx, mid in enumerate(movie_categories)}

    # Read test set only with needed columns for ALS mapping
    test_ratings_slim = pd.read_csv('test_ratings.csv', usecols=['userId', 'movieId', 'rating'])
    test_users = test_ratings_slim['userId'].map(user_to_code)
    test_movies = test_ratings_slim['movieId'].map(movie_to_code)

    valid_mask = test_users.notna() & test_movies.notna()
    valid_users = test_users[valid_mask].astype(int).values
    valid_movies = test_movies[valid_mask].astype(int).values
    als_y_true = test_ratings_slim['rating'][valid_mask].values
    als_y_eval_bin = (als_y_true >= 4.0).astype(int)

    if len(als_y_true) > 0:
        user_factors = als_model.user_factors[valid_users]
        item_factors = als_model.item_factors[valid_movies]
        als_y_pred = np.sum(user_factors * item_factors, axis=1)
        als_y_pred = np.clip(als_y_pred, 0.5, 5.0)

        als_rmse = np.sqrt(mean_squared_error(als_y_true, als_y_pred))
        als_mae = mean_absolute_error(als_y_true, als_y_pred)

        als_true_bin = np.array([1 if r >= 4.0 else 0 for r in als_y_true])
        als_pred_bin = np.array([1 if est >= 4.0 else 0 for est in als_y_pred])
        als_cls = classification_report(als_true_bin, als_pred_bin, output_dict=True, zero_division=0)
        als_roc_auc = roc_auc_score(als_y_eval_bin, als_y_pred)
        als_fpr = calculate_fpr(als_true_bin, als_pred_bin)

        evaluation_summary["ALS (Full Data)"] = {
            "Accuracy": round(als_cls['accuracy'], 4),
            "Precision (Liked)": round(als_cls['1']['precision'], 4),
            "Recall (Liked)": round(als_cls['1']['recall'], 4),
            "F1-Score (Liked)": round(als_cls['1']['f1-score'], 4),
            "ROC-AUC": round(als_roc_auc, 4),
            "FPR": round(als_fpr, 4),
            "RMSE": round(als_rmse, 4),
            "MAE": round(als_mae, 4)
        }
    else:
        evaluation_summary["ALS (Full Data)"] = {"Status": "No overlapping indices"}

    # Deep purge
    del als_data, als_model, user_categories, movie_categories, user_to_code, movie_to_code
    del test_ratings_slim, test_users, test_movies, valid_mask, valid_users, valid_movies, als_y_true, als_y_pred
    gc.collect()
else:
    print("ALS pkl or test dataset missing.")

# -------------------------------------------------------------
# 3. EVALUATE SLIM COLLABORATIVE FILTERING (HIGHLY VECTORIZED)
# -------------------------------------------------------------
print("\n--- [Step 3/3] Evaluating SLIM Model ---")
if os.path.exists("slim_weights.csv") and os.path.exists("slim_user_map.csv") and os.path.exists("slim_movie_map.csv") and os.path.exists('test_ratings.csv'):
    print("Loading saved SLIM model weights...")
    W = pd.read_csv("slim_weights.csv", header=None).values
    user_map_df = pd.read_csv("slim_user_map.csv")
    movie_map_df = pd.read_csv("slim_movie_map.csv")

    user_map = dict(zip(user_map_df['userId'], user_map_df['code']))
    movie_map = dict(zip(movie_map_df['movieId'], movie_map_df['code']))
    del user_map_df, movie_map_df

    print("Reconstructing train interaction reference matrix lazily...")
    train_ratings_slim = pd.read_csv('train_ratings.csv', usecols=['userId', 'movieId', 'rating'])
    train_ratings_slim['u_code'] = train_ratings_slim['userId'].map(user_map)
    train_ratings_slim['m_code'] = train_ratings_slim['movieId'].map(movie_map)
    train_ratings_slim = train_ratings_slim.dropna()

    user_item_matrix = csr_matrix(
        (train_ratings_slim['rating'], (train_ratings_slim['u_code'].astype(int), train_ratings_slim['m_code'].astype(int))),
        shape=(len(user_map), len(movie_map))
    ).tocsr() # Convert to CSR for ultra-fast row slicing
    del train_ratings_slim
    gc.collect()

    print("Loading test ratings for SLIM evaluation...")
    test_ratings_slim = pd.read_csv('test_ratings.csv', usecols=['userId', 'movieId', 'rating'])
    test_ratings_filtered = test_ratings_slim[test_ratings_slim['userId'].isin(user_map) & test_ratings_slim['movieId'].isin(movie_map)]

    # Map user and item IDs to codes
    test_u_codes = test_ratings_filtered['userId'].map(user_map).values.astype(int)
    test_m_codes = test_ratings_filtered['movieId'].map(movie_map).values.astype(int)
    slim_y_true = test_ratings_filtered['rating'].values

    max_items_to_train = W.shape[0]

    # Filter to indices within the trained weights bound (items < 500)
    valid_idx = test_m_codes < max_items_to_train
    test_u_codes = test_u_codes[valid_idx]
    test_m_codes = test_m_codes[valid_idx]
    slim_y_true = slim_y_true[valid_idx]

    print(f"Calculating predictions on {len(slim_y_true):,} test ratings using vectorized vector products...")

    # Pull full sparse row arrays for evaluation users
    # CSR representation allows instant lookup of entire vectors
    predictions = []
    for u, m in zip(test_u_codes, test_m_codes):
        # Fast row slice
        row_start = user_item_matrix.indptr[u]
        row_end = user_item_matrix.indptr[u + 1]

        row_indices = user_item_matrix.indices[row_start:row_end]
        row_data = user_item_matrix.data[row_start:row_end]

        # Fast lookup on SLIM weights
        pred_val = np.dot(row_data, W[m, row_indices])
        predictions.append(pred_val)

    slim_y_pred = np.array(predictions)

    # Exclude ratings that scored non-positive (unpredicted items)
    positive_mask = slim_y_pred > 0
    slim_y_true = slim_y_true[positive_mask]
    slim_y_pred = slim_y_pred[positive_mask]

    if len(slim_y_true) > 0:
        slim_rmse = np.sqrt(mean_squared_error(slim_y_true, slim_y_pred))
        slim_mae = mean_absolute_error(slim_y_true, slim_y_pred)

        slim_true_bin = (slim_y_true >= 4.0).astype(int)
        slim_pred_bin = (slim_y_pred >= 4.0).astype(int)
        slim_cls = classification_report(slim_true_bin, slim_pred_bin, output_dict=True, zero_division=0)
        slim_roc_auc = roc_auc_score(slim_true_bin, slim_y_pred)
        slim_fpr = calculate_fpr(slim_true_bin, slim_pred_bin)

        evaluation_summary["SLIM"] = {
            "Accuracy": round(slim_cls['accuracy'], 4),
            "Precision (Liked)": round(slim_cls['1']['precision'], 4),
            "Recall (Liked)": round(slim_cls['1']['recall'], 4),
            "F1-Score (Liked)": round(slim_cls['1']['f1-score'], 4),
            "ROC-AUC": round(slim_roc_auc, 4),
            "FPR": round(slim_fpr, 4),
            "RMSE": round(slim_rmse, 4),
            "MAE": round(slim_mae, 4)
        }
    else:
        evaluation_summary["SLIM"] = {"Status": "No SLIM overlapping items predicted"}

    del W, user_item_matrix, test_ratings_slim, test_ratings_filtered
    gc.collect()
else:
    print("SLIM artifacts missing.")

# -------------------------------------------------------------
# 4. FINAL COMPARISON REPORT
# -------------------------------------------------------------
print("\n======================================================================")
print("             COMPLETE UNIFIED MODEL PERFORMANCE SUMMARY               ")
print("======================================================================")
summary_df = pd.DataFrame(evaluation_summary).T
print(summary_df.to_string())
print("======================================================================")

=== STARTING MEMORY-OPTIMIZED SERIAL EVALUATION PHASE ===

--- [Step 1/3] Evaluating Content-Based Models in chunks ---
Loading Naive Bayes and Logistic Regression models...
Reading test_ratings.csv in chunks...
Compiling Naive Bayes Metrics...
Compiling Logistic Regression Metrics...

--- [Step 2/3] Evaluating ALS Model ---
Loading ALS Model files...

--- [Step 3/3] Evaluating SLIM Model ---
Loading saved SLIM model weights...
Reconstructing train interaction reference matrix lazily...
Loading test ratings for SLIM evaluation...
Calculating predictions on 1,115,565 test ratings using vectorized vector products...

             COMPLETE UNIFIED MODEL PERFORMANCE SUMMARY               
                     Accuracy  Precision (Liked)  Recall (Liked)  F1-Score (Liked)  ROC-AUC     FPR    RMSE     MAE
Naive Bayes            0.5617             0.5601          0.5592            0.5596   0.5814  0.4359  1.3233  1.1190
Logistic Regression    0.5641             0.5686          0.5176          

Cell-8: ADVANCED METRICS EVALUATION

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import sem, t
from sklearn.metrics import mean_squared_error
import joblib
import gc
import os
from scipy.sparse import csr_matrix

print("=== STARTING INDEPENDENT ADVANCED METRICS EVALUATION FOR ALL FOUR MODELS ===")

# 1. Load ground truth datasets independently
test_ratings = pd.read_csv('test_ratings.csv')
movies_df = pd.read_csv('processed_movies.csv')

total_catalog_movies = movies_df['movieId'].nunique()
total_dataset_users = test_ratings['userId'].nunique()

print(f"Total unique users in test set: {total_dataset_users:,}")
print(f"Total unique movies in catalog: {total_catalog_movies:,}")

# -------------------------------------------------------------
# Helper function to compute sample NDPM
# -------------------------------------------------------------
def compute_ndpm_sample(y_true, y_pred, sample_size=1000):
    """Computes ranking inversion penalty (NDPM proxy) on a sample of user rating pairs."""
    concordant = 0
    discordant = 0
    ties = 0

    if len(y_true) == 0 or len(y_pred) == 0:
        return "N/A"

    indices = np.random.choice(len(y_true), min(sample_size, len(y_true)), replace=False)
    t_sample = np.array(y_true)[indices]
    p_sample = np.array(y_pred)[indices]

    for i in range(len(t_sample)):
        for j in range(i + 1, len(t_sample)):
            if t_sample[i] != t_sample[j]:
                if (t_sample[i] > t_sample[j] and p_sample[i] > p_sample[j]) or \
                   (t_sample[i] < t_sample[j] and p_sample[i] < p_sample[j]):
                    concordant += 1
                elif t_sample[i] == p_sample[i] or t_sample[j] == p_sample[j]:
                    ties += 1
                else:
                    discordant += 1

    total_pairs = concordant + discordant + ties
    if total_pairs == 0:
        return 0.0
    return discordant / total_pairs

# -------------------------------------------------------------
# Helper function to calculate 95% Confidence Interval for residuals
# -------------------------------------------------------------
def get_residuals_confidence_interval(y_true, y_pred):
    if len(y_true) == 0 or len(y_pred) == 0 or len(y_true) != len(y_pred):
        return "N/A", "N/A"
    residuals = np.array(y_true) - np.array(y_pred)
    n = len(residuals)
    se = sem(residuals)
    ci = t.interval(0.95, n - 1, loc=np.mean(residuals), scale=se)
    std_resid = np.std(residuals)
    margin_of_error = 1.96 * std_resid
    return f"({ci[0]:.4f}, {ci[1]:.4f})", f"\u00b1{margin_of_error:.4f}"

advanced_results = {}

# -------------------------------------------------------------
# 1. INDEPENDENT EVALUATION: NAIVE BAYES
# -------------------------------------------------------------
if os.path.exists('nb_model.pkl'):
    print("\nEvaluating Naive Bayes...")
    nb_model = joblib.load('nb_model.pkl')
    genre_columns = [
        '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    merged_test = pd.merge(test_ratings, movies_df, on='movieId')
    X_test = merged_test[genre_columns]
    y_true_all = merged_test['rating'].values

    # Generate predictions independently
    try:
        probs = nb_model.predict_proba(X_test)[:, 1]
        preds = 0.5 + (probs * 4.5)
        nb_ci, nb_margin = get_residuals_confidence_interval(y_true_all, preds)
        nb_rec_items = merged_test['movieId'][preds >= 4.0].nunique()
        nb_coverage = (nb_rec_items / total_catalog_movies) * 100
        nb_ndpm = compute_ndpm_sample(y_true_all, preds)

        advanced_results["Naive Bayes"] = {
            "95% Residual CI": nb_ci,
            "Prediction Interval Margin": nb_margin,
            "Catalog Coverage (%)": f"{nb_coverage:.2f}%",
            "NDPM Inversion Error": round(nb_ndpm, 4) if isinstance(nb_ndpm, float) else nb_ndpm
        }
    except Exception as e:
        print(f"Naive Bayes evaluation failed: {e}")

    del nb_model, merged_test, X_test, y_true_all
    gc.collect()

# -------------------------------------------------------------
# 2. INDEPENDENT EVALUATION: LOGISTIC REGRESSION
# -------------------------------------------------------------
if os.path.exists('lr_sgd.pkl'):
    print("Evaluating Logistic Regression...")
    lr_sgd = joblib.load('lr_sgd.pkl')
    genre_columns = [
        '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]
    merged_test = pd.merge(test_ratings, movies_df, on='movieId')
    X_test = merged_test[genre_columns]
    y_true_all = merged_test['rating'].values

    try:
        probs = lr_sgd.predict_proba(X_test)[:, 1]
        preds = 0.5 + (probs * 4.5)
        lr_ci, lr_margin = get_residuals_confidence_interval(y_true_all, preds)
        lr_rec_items = merged_test['movieId'][preds >= 4.0].nunique()
        lr_coverage = (lr_rec_items / total_catalog_movies) * 100
        lr_ndpm = compute_ndpm_sample(y_true_all, preds)

        advanced_results["Logistic Regression"] = {
            "95% Residual CI": lr_ci,
            "Prediction Interval Margin": lr_margin,
            "Catalog Coverage (%)": f"{lr_coverage:.2f}%",
            "NDPM Inversion Error": round(lr_ndpm, 4) if isinstance(lr_ndpm, float) else lr_ndpm
        }
    except Exception as e:
        print(f"Logistic Regression evaluation failed: {e}")

    del lr_sgd, merged_test, X_test, y_true_all
    gc.collect()

# -------------------------------------------------------------
# 3. INDEPENDENT EVALUATION: ALS COLLABORATIVE FILTERING
# -------------------------------------------------------------
if os.path.exists('als_model.pkl'):
    print("Evaluating ALS Model...")
    als_data = joblib.load('als_model.pkl')
    als_model = als_data['model']
    user_categories = als_data['user_categories']
    movie_categories = als_data['movie_categories']

    user_to_code = {uid: idx for idx, uid in enumerate(user_categories)}
    movie_to_code = {mid: idx for idx, mid in enumerate(movie_categories)}

    test_users = test_ratings['userId'].map(user_to_code)
    test_movies = test_ratings['movieId'].map(movie_to_code)

    valid_mask = test_users.notna() & test_movies.notna()
    valid_users = test_users[valid_mask].astype(int).values
    valid_movies = test_movies[valid_mask].astype(int).values
    als_y_true = test_ratings['rating'][valid_mask].values

    if len(als_y_true) > 0:
        try:
            user_factors = als_model.user_factors[valid_users]
            item_factors = als_model.item_factors[valid_movies]
            als_y_pred = np.sum(user_factors * item_factors, axis=1)
            als_y_pred = np.clip(als_y_pred, 0.5, 5.0)

            als_ci, als_margin = get_residuals_confidence_interval(als_y_true, als_y_pred)
            # Calculate coverage
            als_rec_items = len(np.unique(test_ratings['movieId'].values[valid_mask][als_y_pred >= 4.0]))
            als_coverage = (als_rec_items / total_catalog_movies) * 100
            als_ndpm = compute_ndpm_sample(als_y_true, als_y_pred)

            advanced_results["ALS (Full Data)"] = {
                "95% Residual CI": als_ci,
                "Prediction Interval Margin": als_margin,
                "Catalog Coverage (%)": f"{als_coverage:.2f}%",
                "NDPM Inversion Error": round(als_ndpm, 4) if isinstance(als_ndpm, float) else als_ndpm
            }
        except Exception as e:
            print(f"ALS evaluation failed: {e}")

    del als_data, als_model, user_categories, movie_categories, user_to_code, movie_to_code
    del test_users, test_movies, valid_mask, valid_users, valid_movies, als_y_true, als_y_pred
    gc.collect()

# -------------------------------------------------------------
# 4. INDEPENDENT EVALUATION: SLIM COLLABORATIVE FILTERING
# -------------------------------------------------------------
if os.path.exists("slim_weights.csv") and os.path.exists("slim_user_map.csv") and os.path.exists("slim_movie_map.csv"):
    print("Evaluating SLIM Model...")
    W = pd.read_csv("slim_weights.csv", header=None).values
    user_map_df = pd.read_csv("slim_user_map.csv")
    movie_map_df = pd.read_csv("slim_movie_map.csv")

    user_map = dict(zip(user_map_df['userId'], user_map_df['code']))
    movie_map = dict(zip(movie_map_df['movieId'], movie_map_df['code']))
    del user_map_df, movie_map_df

    train_ratings_slim = pd.read_csv('train_ratings.csv', usecols=['userId', 'movieId', 'rating'])
    train_ratings_slim['u_code'] = train_ratings_slim['userId'].map(user_map)
    train_ratings_slim['m_code'] = train_ratings_slim['movieId'].map(movie_map)
    train_ratings_slim = train_ratings_slim.dropna()

    user_item_matrix = csr_matrix(
        (train_ratings_slim['rating'], (train_ratings_slim['u_code'].astype(int), train_ratings_slim['m_code'].astype(int))),
        shape=(len(user_map), len(movie_map))
    ).tocsr()
    del train_ratings_slim

    test_ratings_filtered = test_ratings[test_ratings['userId'].isin(user_map) & test_ratings['movieId'].isin(movie_map)]
    test_u_codes_all = test_ratings_filtered['userId'].map(user_map).values.astype(int)
    test_m_codes_all = test_ratings_filtered['movieId'].map(movie_map).values.astype(int)
    slim_y_true_all = test_ratings_filtered['rating'].values

    max_items_to_train = W.shape[0]
    valid_idx = test_m_codes_all < max_items_to_train

    test_u_codes = test_u_codes_all[valid_idx]
    test_m_codes = test_m_codes_all[valid_idx]
    slim_y_true = slim_y_true_all[valid_idx]

    predictions = []
    for u, m in zip(test_u_codes, test_m_codes):
        row_start = user_item_matrix.indptr[u]
        row_end = user_item_matrix.indptr[u + 1]
        row_indices = user_item_matrix.indices[row_start:row_end]
        row_data = user_item_matrix.data[row_start:row_end]
        pred_val = np.dot(row_data, W[m, row_indices])
        predictions.append(pred_val)

    slim_y_pred = np.array(predictions)
    positive_mask = slim_y_pred > 0

    slim_y_true_aligned = slim_y_true[positive_mask]
    slim_y_pred_aligned = slim_y_pred[positive_mask]
    aligned_m_codes = test_m_codes[positive_mask]

    if len(slim_y_true_aligned) > 0:
        slim_ci, slim_margin = get_residuals_confidence_interval(slim_y_true_aligned, slim_y_pred_aligned)
        slim_rec_items = len(np.unique(aligned_m_codes[slim_y_pred_aligned >= 4.0]))
        slim_coverage = (slim_rec_items / total_catalog_movies) * 100
        slim_ndpm = compute_ndpm_sample(slim_y_true_aligned, slim_y_pred_aligned)

        advanced_results["SLIM"] = {
            "95% Residual CI": slim_ci,
            "Prediction Interval Margin": slim_margin,
            "Catalog Coverage (%)": f"{slim_coverage:.2f}%",
            "NDPM Inversion Error": round(slim_ndpm, 4) if isinstance(slim_ndpm, float) else slim_ndpm
        }

    del W, user_item_matrix, test_ratings_filtered, test_u_codes, test_m_codes, slim_y_true, slim_y_pred
    gc.collect()

# -----------------------------------------------------
# DISPLAY COMPREHENSIVE ADVANCED REPORT
# -----------------------------------------------------
print("\n======================================================================")
print("                COMPREHENSIVE ADVANCED METRICS REPORT                 ")
print("======================================================================")
if advanced_results:
    adv_df = pd.DataFrame(advanced_results).T
    print(adv_df.to_string())
else:
    print("No model run data found on disk! Please ensure model artifacts are in workspace.")
print("======================================================================")

gc.collect()


=== STARTING INDEPENDENT ADVANCED METRICS EVALUATION FOR ALL FOUR MODELS ===
Total unique users in test set: 162,538
Total unique movies in catalog: 62,423

Evaluating Naive Bayes...
Evaluating Logistic Regression...
Evaluating ALS Model...
Evaluating SLIM Model...

                COMPREHENSIVE ADVANCED METRICS REPORT                 
                      95% Residual CI Prediction Interval Margin Catalog Coverage (%) NDPM Inversion Error
Naive Bayes          (0.7921, 0.7936)                    ±2.0767                0.03%                0.451
Logistic Regression  (0.8002, 0.8016)                    ±2.0674                0.00%               0.4741
ALS (Full Data)      (2.9413, 2.9427)                    ±2.0449                0.02%               0.5946
SLIM                 (2.3894, 2.3937)                    ±2.2393                0.15%               0.3518


0

In [ ]:
# === OPTIMIZATION SWEEP PLAYGROUND ===
# This code performs hyperparameter search for content-based models and outlines suggestions for collaborative filtering.

import numpy as np
import pandas as pd
import joblib
import os
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score, mean_squared_error

print("=== OPTIMIZATION SWEEP PLAYGROUND ===")

# 1. OPTIMIZATION FOR LOGISTIC REGRESSION (SGD CLASSIFIER)
# Let's sweep alpha values on a smaller sample to find a better ROC-AUC score
if os.path.exists('train_ratings.csv') and os.path.exists('processed_movies.csv'):
    print("\n--- Running Hyperparameter Sweep for SGD Logistic Regression ---")

    # Load sample data to keep sweep fast and memory safe
    train_sample = pd.read_csv('train_ratings.csv', nrows=500000)
    test_sample = pd.read_csv('test_ratings.csv', nrows=200000)
    movies_df = pd.read_csv('processed_movies.csv')

    genre_columns = [
        '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
        'Thriller', 'War', 'Western'
    ]

    train_merged = pd.merge(train_sample, movies_df, on='movieId')
    train_merged['liked'] = (train_merged['rating'] >= 4.0).astype(int)
    X_train = train_merged[genre_columns]
    y_train = train_merged['liked']

    test_merged = pd.merge(test_sample, movies_df, on='movieId')
    test_merged['liked'] = (test_merged['rating'] >= 4.0).astype(int)
    X_test = test_merged[genre_columns]
    y_test = test_merged['liked']

    alphas = [1e-5, 1e-4, 1e-3, 1e-2]
    best_alpha = None
    best_auc = 0

    for alpha in alphas:
        # Test both L2 and ElasticNet penalties
        for penalty in ['l2', 'elasticnet']:
            model = SGDClassifier(loss='log_loss', penalty=penalty, alpha=alpha, l1_ratio=0.15, random_state=42)
            model.fit(X_train, y_train)

            probs = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, probs)
            print(f"Alpha: {alpha:<6} | Penalty: {penalty:<10} | Test ROC-AUC: {auc:.4f}")

            if auc > best_auc:
                best_auc = auc
                best_alpha = (alpha, penalty)

    print(f"\nOptimized SGD Recommendation: Use Alpha={best_alpha[0]} with Penalty='{best_alpha[1]}' (AUC: {best_auc:.4f})")

# 2. SUGGESTED OPTIMIZATIONS FOR COLLABORATIVE FILTERING MODELS
print("\n--- Recommended Optimization Parameters for CF Models ---")
print("""
For ALS:
- Increase 'factors' to 64 or 128 (captures richer latent profiles)
- Lower 'regularization' to 0.01 if the model is underfitting, or increase to 0.1 if overfitting
- Tune 'iterations' (15 to 25) to allow latent profiles to fully converge

For SLIM:
- Decrease ElasticNet 'alpha' to 0.01 to allow more catalog coverage
- Fine-tune 'l1_ratio' closer to 0.1 to allow slightly more non-zero weights for candidate discovery
""")